In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets pillow trl kagglehub

In [ ]:
import os
import json
import torch
import kagglehub

from PIL import Image
from datasets import Dataset

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model
)

In [ ]:
dataset_path = kagglehub.dataset_download(
    "parthplc/facebook-hateful-meme-dataset"
)

print("Dataset Path:", dataset_path)

In [ ]:
#Load Dataset Paths

TRAIN_FILE = os.path.join(
    dataset_path,
    "data/train.jsonl"
)

DEV_FILE = os.path.join(
    dataset_path,
    "data/dev.jsonl"
)

TEST_FILE = os.path.join(
    dataset_path,
    "data/test.jsonl"
)

IMAGE_DIR = os.path.join(
    dataset_path,
    "data"
)

print(TRAIN_FILE)
print(DEV_FILE)
print(TEST_FILE)

In [ ]:
# ==========================================
# LOAD ONLY FIRST 500 SAMPLES
# ==========================================

def load_jsonl(file_path, limit=None):

    data = []

    with open(file_path, "r") as f:

        for idx, line in enumerate(f):

            if limit and idx >= limit:
                break

            sample = json.loads(line)

            data.append(sample)

    return data


train_data = load_jsonl(
    TRAIN_FILE
)

dev_data = load_jsonl(
    DEV_FILE
)

test_data = load_jsonl(
    TEST_FILE
)

print("TRAIN:", len(train_data))
print("DEV:", len(dev_data))
print("TEST:", len(test_data))

In [ ]:
def format_sample(sample):

    image_path = os.path.join(IMAGE_DIR, sample["img"])

    # ✅ FIX LABEL HERE (0/1 ONLY)
    label = int(sample["label"])

    prompt = """
Analyze the meme.

Classify it as:
- 0 = non-hateful
- 1 = hateful

Answer only 0 or 1.
"""

    return {
        "image": image_path,
        "prompt": prompt,
        "label": label
    }

train_formatted = [format_sample(x) for x in train_data]
dev_formatted   = [format_sample(x) for x in dev_data]
test_formatted  = [format_sample(x) for x in dev_data]

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
def preprocess(example):

    image = Image.open(example["image"]).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": example["prompt"]}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {k: v.squeeze(0) for k, v in inputs.items()}

    # ✅ FIXED LABEL (CRITICAL)
    inputs["labels"] = torch.tensor(example["label"], dtype=torch.long)

    return inputs

In [ ]:
train_dataset = Dataset.from_list(
    train_formatted
)

dev_dataset = Dataset.from_list(
    dev_formatted
)

In [ ]:
def preprocess(example):

    image = Image.open(example["image"]).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": example["prompt"]}
            ]
        }
    ]

    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_inputs = processor(
        text=[prompt_text],
        images=[image],
        return_tensors="pt"
    )

    prompt_inputs = {k: v.squeeze(0) for k, v in prompt_inputs.items()}

    # ✅ build answer text
    answer_text = "hateful" if example["label"] == 1 else "non-hateful"

    answer_tokens = processor.tokenizer(
        answer_text,
        add_special_tokens=False
    ).input_ids

    # prompt length
    input_ids = prompt_inputs["input_ids"].tolist()
    attention_mask = prompt_inputs["attention_mask"].tolist()

    # append answer
    input_ids = input_ids + answer_tokens
    attention_mask = attention_mask + [1] * len(answer_tokens)

    # labels (-100 for prompt, tokens for answer)
    labels = [-100] * len(prompt_inputs["input_ids"]) + answer_tokens

    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
        "pixel_values": prompt_inputs["pixel_values"]
    }

In [ ]:
train_dataset = train_dataset.map(
    preprocess
)

dev_dataset = dev_dataset.map(
    preprocess
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_hatememes",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    num_train_epochs=10,

    logging_steps=10,

    save_strategy="epoch",

    fp16=True,

    remove_unused_columns=False,

    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

In [ ]:
trainer.train()

In [ ]:
#model.save_pretrained(
    #"./qwen_hatememes_lora"
#)

#processor.save_pretrained(
    #"./qwen_hatememes_lora"
#)

In [ ]:
image = Image.open(
    os.path.join(
        IMAGE_DIR,
        test_data[0]["img"]
    )
).convert("RGB")

prompt = """
Analyze this meme.

Classify it as:
- hateful
- non-hateful

Answer only one word.
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    images=[image],
    return_tensors="pt"
).to(model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=20
)

generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(
        inputs.input_ids,
        generated_ids
    )
]

output = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True
)

print(output[0])

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

In [ ]:
def predict(example):

    image = Image.open(example["image"]).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": example["prompt"]}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5)

    decoded = processor.batch_decode(
        output,
        skip_special_tokens=True
    )[0].lower()

    # convert text → label
    if "hateful" in decoded:
        return 1
    else:
        return 0

In [ ]:
y_true = []
y_pred = []

for sample in dev_data:

    label = int(sample["label"])
    pred = predict({
        "image": os.path.join(IMAGE_DIR, sample["img"]),
        "prompt": """
Classify meme as:
0 = non-hateful
1 = hateful
Answer only 0 or 1.
"""
    })

    y_true.append(label)
    y_pred.append(pred)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="binary"
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
# ============================================================
# FINAL EVALUATION
# ============================================================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix
)

accuracy = accuracy_score(

    y_true,

    y_pred
)

precision = precision_score(

    y_true,

    y_pred
)

recall = recall_score(

    y_true,

    y_pred
)

f1 = f1_score(

    y_true,

    y_pred
)

avg_time = np.mean(

    inference_times
)

avg_energy = np.mean(

    energy_consumptions
)

report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Hateful",

        "Hateful"
    ],

    digits=4
)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1-score:", f1)

print("Inference Time:", avg_time)

print("Energy (kWh):", avg_energy)

print()

print("Classification Report:")

print(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
y_score = []

for sample in dev_data:

    image = Image.open(os.path.join(IMAGE_DIR, sample["img"])).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Answer: hateful or non-hateful"}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False)

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5)

    decoded = processor.batch_decode(output, skip_special_tokens=True)[0].lower()

    score = 1 if "hateful" in decoded else 0
    y_score.append(score)

In [ ]:
roc = roc_auc_score(y_true, y_score)
print("ROC-AUC:", roc)

In [ ]:
import time

start = time.time()

for i in range(len(dev_data)):
    _ = predict({
        "image": os.path.join(IMAGE_DIR, dev_data[i]["img"]),
        "prompt": "Classify meme as 0 or 1"
    })

end = time.time()

avg_time = (end - start) / len(dev_data)
print("Avg inference time (sec/image):", avg_time)

In [ ]:
print("\n===== FINAL RESULTS =====")

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("ROC-AUC:", roc)
print("Confusion Matrix:\n", cm)
print("Avg Inference Time:", avg_time, "sec/image")